# 01 — Business Understanding and Exploratory Data Analysis

**Capstone Project 24.1 | Enterprise SaaS Churn Risk**

## Business problem
Churn and downgrade can reduce recurring revenue and customer lifetime value. Customer Success teams need a repeatable way to identify accounts for proactive review. This project asks whether account profile, product adoption, engagement and value-realization indicators can distinguish churn outcomes.

> **Data transparency:** This repository uses a synthetic teaching dataset of 3,000 simulated SaaS accounts. It is not customer telemetry. The target is generated from an explicit simulation mechanism involving the same broad risk indicators studied in this notebook. Results are educational demonstrations, not real-world predictions or industry benchmarks.

In [1]:
from pathlib import Path
import runpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data/raw/enterprise_saas_churn_synthetic.csv'
if not DATA.exists():
    runpy.run_path(str(ROOT / 'scripts/generate_synthetic_dataset.py'))
df = pd.read_csv(DATA)
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

Dataset: 3,000 rows × 18 columns


,account_id,industry,region,customer_tier,tenure_months,annual_recurring_revenue_usd,licensed_seats,license_utilization_pct,dau_mau_ratio,active_modules,monthly_api_calls,qbrs_attended_last_4,csat_score,kpi_milestones_achieved_pct,automated_workflows,support_cases_last_90d,days_to_renewal,churned
0,SAAS-00001,Manufacturing,APAC,Mid-Market,62,155525,173,41.5,0.653,2,10072.0,2.0,1.0,48.6,0,5,5,1
1,SAAS-00002,Retail,Europe,Enterprise,74,2066138,2101,48.6,0.358,4,NaN,0.0,2.0,26.9,51,3,64,1
2,SAAS-00003,Manufacturing,North America,Mid-Market,14,316387,202,56.8,0.477,5,6216.0,4.0,NaN,30.4,30,5,52,0
3,SAAS-00004,Technology,APAC,Enterprise,23,615348,485,82.0,0.907,6,9619.0,0.0,4.0,34.5,64,3,5,0
4,SAAS-00005,Financial Services,North America,Enterprise,6,1815299,1468,77.5,0.880,4,10286.0,1.0,5.0,58.9,30,4,121,0


## 1. Data quality and target balance

The account identifier is excluded from modeling. Missing values are retained for pipeline-based imputation. Duplicate account IDs are checked before analysis.

In [2]:
quality = pd.DataFrame({'missing_count':df.isna().sum(),'missing_pct':(df.isna().mean()*100).round(2),'unique_values':df.nunique()})
display(quality.sort_values('missing_pct',ascending=False))
print('Duplicate account IDs:',df.account_id.duplicated().sum())
display(df.churned.value_counts().rename(index={0:'Retained',1:'Churned'}).to_frame('accounts'))
print(f'Simulated churn prevalence: {df.churned.mean():.1%}')

,missing_count,missing_pct,unique_values
csat_score,75,2.5,5
kpi_milestones_achieved_pct,75,2.5,814
qbrs_attended_last_4,75,2.5,5
monthly_api_calls,75,2.5,2537
industry,0,0.0,6
account_id,0,0.0,3000
region,0,0.0,4
customer_tier,0,0.0,3
tenure_months,0,0.0,82
annual_recurring_revenue_usd,0,0.0,2992


Duplicate account IDs: 0


,accounts
churned,
Retained,2542
Churned,458


Simulated churn prevalence: 15.3%


## 2. Categorical and continuous exploration

The following charts summarize class balance, churn rate by customer tier and the distribution of important numeric variables. Unusual enterprise ARR or usage values are not automatically removed because they may be legitimate.

In [3]:
IMG = ROOT / 'images'; IMG.mkdir(exist_ok=True)
fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.countplot(x=df.churned.astype(str),color='steelblue',ax=axes[0])
axes[0].set(title='Simulated account outcome distribution',xlabel='Churn outcome (0 retained, 1 churned)',ylabel='Accounts')
tier_rate=df.groupby('customer_tier',observed=True).churned.mean().reset_index()
sns.barplot(data=tier_rate,x='customer_tier',y='churned',color='slateblue',ax=axes[1])
axes[1].set(title='Simulated churn rate by tier',xlabel='Customer tier',ylabel='Share churned',ylim=(0,1))
plt.tight_layout(); plt.savefig(IMG/'eda_target_and_tier.png',dpi=160); plt.show()
features=['tenure_months','annual_recurring_revenue_usd','license_utilization_pct','dau_mau_ratio','active_modules','support_cases_last_90d','days_to_renewal']
fig,axes=plt.subplots(3,3,figsize=(14,10)); axes=axes.flatten()
for ax,col in zip(axes,features):
    sns.histplot(data=df,x=col,hue='churned',stat='density',common_norm=False,element='step',ax=ax)
    ax.set_title(col.replace('_',' ').title())
for ax in axes[len(features):]: ax.remove()
plt.tight_layout(); plt.savefig(IMG/'eda_numeric_distributions.png',dpi=160); plt.show()

In [4]:
corr_cols=['tenure_months','annual_recurring_revenue_usd','license_utilization_pct','dau_mau_ratio','active_modules','monthly_api_calls','qbrs_attended_last_4','csat_score','kpi_milestones_achieved_pct','automated_workflows','support_cases_last_90d','days_to_renewal','churned']
fig,ax=plt.subplots(figsize=(12,9))
sns.heatmap(df[corr_cols].corr(numeric_only=True),cmap='vlag',center=0,ax=ax)
ax.set_title('Numeric feature correlation matrix')
plt.tight_layout(); plt.savefig(IMG/'eda_correlation_heatmap.png',dpi=160); plt.show()
for col in ['customer_tier','industry','region']:
    display(df.groupby(col,observed=True).churned.agg(accounts='size',churn_rate='mean').sort_values('churn_rate',ascending=False))

,accounts,churn_rate
customer_tier,,
SMB,728,0.189560
Enterprise,1027,0.145083
Mid-Market,1245,0.137349


,accounts,churn_rate
industry,,
Technology,625,0.163200
Manufacturing,440,0.159091
Financial Services,557,0.157989
Healthcare,474,0.151899
Retail,541,0.142329
Telecom,363,0.134986


,accounts,churn_rate
region,,
Europe,821,0.160780
APAC,677,0.155096
North America,1296,0.147377
LATAM,206,0.145631


## 3. Initial interpretation and limitations

The simulated label mechanism deliberately makes lower adoption, weaker engagement, lower satisfaction/value realization and higher support burden associated with greater churn probability. This lets the modeling workflow demonstrate signal recovery. It does **not** prove that these relationships or rates hold in a real SaaS portfolio. Notebook 03 evaluates how well classifiers recover the simulated outcome on held-out records.